# 01 - Corpus and coverage

What is in TGDataset2, how deep it goes, and where it stops. This notebook backs
the corpus description in the thesis: the headline scalars, the like-for-like
comparison against the predecessor corpus, the per-channel depth distribution,
the daily volume series decomposed into its extensive and intensive margins, the
posting-rhythm heatmap, and the audit of what the crawler saw but could not
follow.

**Figures produced here**

| Output | Thesis figure |
|---|---|
| `channel_message_dist` | per-channel message-count distribution |
| `daily_messages_normalised` | daily volume, decomposed |
| `activity_heatmap` | posting activity by hour and weekday |
| `frontier_link_types` | `t.me` reference types in collected content |

**Tables produced here**: corpus scalars, the dataset comparison and its additive
companion, and the corrected forward ratio.

**Running this.** The database is not shipped with this repository (see
`../README.md`). Sections marked *offline* read a CSV from `../data/` and run
anywhere; the rest need `TGDATASET2_DB_URL` pointing at a Postgres instance with
the TGDataset2 schema (`../docs/schema.md`). The SQL is the record of how each
number was computed, so it is carried here verbatim whether or not it can run.

In [ ]:
import math

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns

from _shared import DATA, FIGURES, csv, q, setup_plots, shape

setup_plots()

## 1. Headline scalars

One query for every top-level count in the corpus description. Report these as
shares with an as-of date rather than as bare totals: the database was still
ingesting while the analysis ran, so any absolute count is a reading taken at a
moment, not a fixed property of the corpus.

Note what the `channels` count is and is not. It counts every channel *row*,
including the ones discovered by the snowball but never scraped, so it is far
larger than the number of channels that carry messages. The corpus is deep, not
wide: only a small fraction of known channels were actually crawled.

In [ ]:
SQL_SCALARS = """
    SELECT (SELECT count(*) FROM channels)                                        AS channels,
           (SELECT count(*) FROM channels WHERE linked_to_channel_id IS NULL)     AS broadcast_channels,
           (SELECT count(*) FROM channels WHERE linked_to_channel_id IS NOT NULL) AS discussion_groups,
           (SELECT count(*) FROM messages)                                        AS messages,
           (SELECT count(*) FROM metadata_snapshots)                              AS snapshots,
           (SELECT count(*) FROM message_edits)                                   AS edits,
           (SELECT count(*) FROM polls)                                           AS polls,
           (SELECT count(*) FROM profile_pictures)                                AS profile_pictures,
           (SELECT count(*) FROM unresolved_references
             WHERE NOT invalid AND resolved_at IS NULL)                           AS pending_refs,
           (SELECT min(date)::date FROM messages)                                 AS first_message,
           (SELECT max(date)::date FROM messages)                                 AS last_message
"""

# Offline path: the shipped CSV is the frozen result of exactly this query.
scalars = csv("corpus_scalars.csv", index_col=0)
scalars.rename(columns={scalars.columns[0]: "value"}, inplace=True)
scalars.assign(display=scalars["value"].map(
    lambda v: f"{int(v):,}" if str(v).isdigit() else str(v)))[["display"]]

## 2. The like-for-like comparison

The predecessor corpus, TGDataset (La Morgia et al., KDD 2025), reports 498.3 M
messages. That is a count of **channel** messages. TGDataset2 also crawls each
broadcast channel's linked discussion group, whose rows are comments and replies
rather than channel posts, so a total-against-total comparison would flatter this
corpus by counting a different kind of record. The comparison below is
channel-only against channel-only; the comment, edit, deletion, snapshot and poll
layers are listed separately as additive contributions and never summed into the
message total.

**Orientation of the linked-group relation.** A `channels` row with
`linked_to_channel_id IS NOT NULL` *is itself* the discussion group; the column's
value points at the parent broadcast channel. So broadcast messages are the rows
whose channel has `linked_to_channel_id IS NULL`, and linked-group messages are
the complement. Getting this backwards inverts every comment statistic in the
thesis, so it is worth stating plainly.

In [ ]:
SQL_PER_CHANNEL = """
    SELECT m.channel_id,
           bool_or(c.linked_to_channel_id IS NOT NULL)      AS is_linked_group,
           count(*)                                         AS msg_count,
           count(*) FILTER (WHERE m.is_forwarded)           AS fwd_count,
           count(*) FILTER (WHERE m.is_deleted)             AS deleted_count,
           count(*) FILTER (WHERE m.reactions IS NOT NULL)  AS react_count,
           min(m.date)                                      AS first_msg,
           max(m.date)                                      AS last_msg
    FROM messages m
    JOIN channels c ON c.channel_id = m.channel_id
    GROUP BY m.channel_id
"""

# Predecessor baselines, as published (channel messages).
BASELINES = {
    "TGDataset (KDD '25)": {"channels": 120_979, "channel_msgs": 498_320_597, "to": "Jul 2022"},
    "Pushshift":           {"channels": 27_801,  "channel_msgs": 192_044_689, "to": "-"},
    "Nobari et al.":       {"channels": 2_556,   "channel_msgs": 36_928,      "to": "-"},
}

comparison = csv("dataset_comparison.csv", index_col=0)
ours = comparison.loc["TGDataset2 (ours)"]
pred = BASELINES["TGDataset (KDD '25)"]
ratio = ours["channel_messages"] / pred["channel_msgs"]
depth = (ours["channel_messages"] / ours["channels"]) / (pred["channel_msgs"] / pred["channels"])
print(f"channel-only messages: {ratio:.2f}x the predecessor's 498.3 M (a subset, not a superset)")
print(f"messages per channel : {depth:.1f}x the predecessor's (fewer channels, far deeper each)")
print(f"plus {int(ours['comment_messages']):,} discussion-group rows the predecessor never collected")
comparison

### Additive contributions

Records of a different kind from a channel message, listed rather than summed.
The predecessor is a static July 2022 snapshot of channel messages; these layers
are what repeated revisits buy.

One caveat on the first row. `linked-group comments/replies` counts every row in
a crawled discussion group, which includes in-group chatter that is not anchored
on any channel post. Notebook 02 separates the two and finds that a substantially
smaller share are true comments on posts. Read this row as the size of the
linked-group layer, not as a count of comments.

In [ ]:
SQL_ADDITIVE = """
    SELECT (SELECT count(*) FROM message_edits)       AS edits,
           (SELECT count(*) FROM metadata_snapshots)  AS snapshots,
           (SELECT count(*) FROM polls)               AS polls
"""

additive = csv("additive_contributions.csv", index_col=0)
additive.assign(count=additive["count"].map("{:,}".format))

### The corrected forward ratio

The raw forward ratio (`is_forwarded` over all messages) overstates editorial
forwarding, because a large block of "forwards" are Telegram's own automatic
mirrors: when a broadcast channel has a linked discussion group, every post is
echoed into that group as a forward from the channel itself. Those rows are
duplicates of content already in the corpus, not one channel choosing to
republish another.

Subtracting them requires the forward-edge graph, since an auto-mirror is
identified structurally: the destination is a discussion group whose parent is
the source channel.

In [ ]:
SQL_FWD_EDGES = """
    SELECT fwd_from_channel_id AS src, channel_id AS dst, count(*)::bigint AS weight
    FROM messages
    WHERE is_forwarded AND fwd_from_channel_id IS NOT NULL
    GROUP BY fwd_from_channel_id, channel_id
"""

SQL_FWD_TOTALS = """
    SELECT count(*)::bigint                                          AS total_messages,
           count(*) FILTER (WHERE is_forwarded)::bigint              AS forwarded,
           count(*) FILTER (WHERE is_forwarded
                             AND fwd_from_channel_id IS NOT NULL)::bigint AS traceable_forwarded
    FROM messages
"""

# An edge is an auto-mirror when the destination is a discussion group whose
# `linked_to_channel_id` is the source channel:
#
#     edges["is_auto_mirror"] = edges["dst_is_group"] & (edges["dst"].map(parent) == edges["src"])
#
# where `parent` maps channel_id -> linked_to_channel_id over the node table.

csv("fwd_ratio_corrected.csv")

## 3. Per-channel depth

The corpus is heavy-tailed: a small fraction of channels accounts for most of the
messages. The distribution below is plotted on log-log axes, where that shows up
as a long straight sweep rather than a bell.

This section needs the database. The per-channel count table it plots is 1.26 MB
of channel identifiers and is not shipped, and no smaller summary reproduces the
shape of a distribution.

In [ ]:
SQL_PER_CHANNEL_COUNTS = """
    SELECT channel_id,
           count(*)::bigint                             AS n,
           count(*) FILTER (WHERE is_forwarded)::bigint AS fwd
    FROM messages
    GROUP BY channel_id
"""

counts = q(SQL_PER_CHANNEL_COUNTS)
n = np.sort(counts["n"].to_numpy())
total = int(n.sum())
print(f"channels with at least one message: {len(counts):,}")

# How concentrated? Number of channels needed to reach 50 / 90 / 99 % of messages.
share = np.cumsum(n[::-1]) / total
concentration = pd.DataFrame({
    "% of messages": [50, 90, 99],
    "channels needed": [int(np.searchsorted(share, t) + 1) for t in (0.50, 0.90, 0.99)],
})
concentration["% of channels"] = (concentration["channels needed"] / len(n) * 100).round(3)
display(concentration)

bins = np.logspace(0, math.ceil(math.log10(n.max() + 1)), 60)
fig, ax = plt.subplots(figsize=(5.48, 3.04))
ax.hist(n, bins=bins, color="#4c78a8", edgecolor="white", linewidth=0.4)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Messages per channel (log)")
ax.set_ylabel("Channels (log)")
ax.set_title(f"Per-channel message-count distribution "
             f"(N = {len(n):,} channels with >=1 message)")
fig.tight_layout()
fig.savefig(FIGURES / "channel_message_dist.png", bbox_inches="tight")
fig.savefig(FIGURES / "channel_message_dist.pdf", bbox_inches="tight")
plt.show()

## 4. Daily volume, decomposed (*offline*)

Daily message volume rises over the corpus span, but that rise has two possible
causes and they mean different things. The **extensive** margin is how many
channels are posting at all on a given day, which the snowball inflates by
construction: later days have more channels in the crawl. The **intensive**
margin is how much each active channel posts, which the crawl does not inflate.

The three panels separate them: raw volume, active channels per day, and messages
per active channel. Only the third is a claim about Telegram rather than about
the crawler. Four external events are marked, spanning all three panels so the
eye can link them.

In [ ]:
SQL_DAILY = """
    SELECT day, count(*)::int AS active_channels, sum(n)::bigint AS n_msgs
    FROM (
        SELECT channel_id, date_trunc('day', date)::date AS day, count(*) AS n
        FROM messages
        WHERE date >= '2015-01-01'::timestamptz
          AND date < (now() + interval '1 day')
        GROUP BY channel_id, date_trunc('day', date)::date
    ) cd
    GROUP BY day
    ORDER BY day
"""

daily = csv("daily_volume_normalised.csv", parse_dates=["day"]).set_index("day").sort_index()
print(f"daily rows: {len(daily):,}   "
      f"span: {daily.index.min().date()} to {daily.index.max().date()}")

# Did the intensive margin actually move? Compare an early dense year to a recent one.
for col in ("active_channels", "msgs_per_active"):
    a = float(daily.loc["2017", col].median())
    b = float(daily.loc["2024", col].median())
    print(f"  median {col}: 2017={a:,.1f}  2024={b:,.1f}  ({b / a:.2f}x)")

In [ ]:
MILESTONES = [
    ("2021-01-06", "US Capitol riot"),
    ("2022-02-24", "Russia invades Ukraine"),
    ("2026-01-03", "Maduro capture"),
    ("2026-02-28", "US/Israel-Iran war"),
]
# Offset in points from the top of the event line, horizontal anchor, rotation.
# Steep labels keep each label's horizontal reach short, so the long "Russia
# invades Ukraine" tail clears the 2026 cluster; the two 2026 events are only
# about eight weeks apart, so they splay into a V.
EV_COLOR = "#2e7d32"
EV_PLACE = {
    "US Capitol riot":        dict(dx=1,  dy=2, ha="left",  rot=45),
    "Russia invades Ukraine": dict(dx=1,  dy=2, ha="left",  rot=45),
    "Maduro capture":         dict(dx=-2, dy=4, ha="right", rot=-45),
    "US/Israel-Iran war":     dict(dx=2,  dy=4, ha="left",  rot=45),
}
PANELS = [
    ("n_msgs",          None,                     "Messages per day (log)",        "#1f3b6a", True),
    ("active_channels", "active_channels_roll7",  "Active channels per day (log)", "#2a7f5f", True),
    ("msgs_per_active", "msgs_per_active_roll7",  "Messages per active channel",   "#b5651d", False),
]

fig, axes = plt.subplots(3, 1, figsize=(5.48, 7.0), sharex=True)
for ax, (raw, roll, ylab, color, logy) in zip(axes, PANELS):
    ax.plot(daily.index, daily[raw], color="#cfd8e3", lw=0.7)
    if roll:
        ax.plot(daily.index, daily[roll], color=color, lw=0.8, label="7-day mean")
        ax.legend(loc="upper left", fontsize=7)
    else:
        ax.plot(daily.index, daily[raw].rolling(7, center=True, min_periods=1).mean(),
                color=color, lw=0.8)
    if logy:
        ax.set_yscale("log")
    if ax is not axes[0]:
        ax.set_title(ylab)
axes[0].set_title(PANELS[0][2], loc="left", pad=6)   # freed right side takes the labels

for date_str, label in MILESTONES:
    d = pd.Timestamp(date_str)
    if not (daily.index.min() <= d <= daily.index.max()):
        continue
    for ax in axes:
        ax.axvspan(d - pd.Timedelta(days=7), d + pd.Timedelta(days=7),
                   color=EV_COLOR, alpha=0.10, lw=0, zorder=0)
        ax.axvline(d, color=EV_COLOR, lw=0.8, alpha=0.55, zorder=1)
    p = EV_PLACE[label]
    axes[0].annotate(label, xy=(d, 1.0), xycoords=("data", "axes fraction"),
                     xytext=(p["dx"], p["dy"]), textcoords="offset points",
                     rotation=p["rot"], rotation_mode="anchor",
                     ha=p["ha"], va="bottom", fontsize=7, color=EV_COLOR)

fig.suptitle("Daily volume = active channels (extensive) x msgs/active channel (intensive)", y=0.9)
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(FIGURES / "daily_messages_normalised.png", bbox_inches="tight")
fig.savefig(FIGURES / "daily_messages_normalised.pdf", bbox_inches="tight")
plt.show()

## 5. Posting rhythm

Hour of day against day of week, over the last twelve months of the corpus so it
reflects the current rhythm rather than an average over the whole span.
Timestamps are `timestamptz` and read in UTC, so this is a UTC clock over a
multi-timezone corpus: the shape is a blend of local rhythms, not one of them.
Postgres numbers `dow` from 0 = Sunday.

The result frame is 168 rows, but no CSV of it was kept, so this section needs
the database.

In [ ]:
SQL_HEATMAP = """
    SELECT EXTRACT(dow  FROM date)::int AS dow,
           EXTRACT(hour FROM date)::int AS hour,
           count(*) AS n
    FROM messages
    WHERE date >= (SELECT max(date) FROM messages) - interval '12 months'
    GROUP BY dow, hour
    ORDER BY dow, hour
"""

heat = q(SQL_HEATMAP)
hp = (heat.pivot(index="dow", columns="hour", values="n")
          .reindex(index=range(7), columns=range(24), fill_value=0))
hp.index = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

fig, ax = plt.subplots(figsize=(5.48, 1.74))
sns.heatmap(hp, cmap="viridis", ax=ax, cbar_kws={"label": "messages"})
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_title("Posting activity by hour x day-of-week (last 12 months, UTC)")
ax.set_xlabel("Hour")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(FIGURES / "activity_heatmap.png", bbox_inches="tight")
fig.savefig(FIGURES / "activity_heatmap.pdf", bbox_inches="tight")
plt.show()

## 6. The frontier: what the crawler saw but could not follow (*offline*)

The snowball discovers new channels from `t.me` references inside collected
content. Not every reference is followable, and the ones that are not are a
structural boundary of the corpus rather than a gap that more crawling would
close.

Each `t.me` URL is classified by what it points at:

- **`invite(+hash)`** and **`invite(joinchat)`**: private-group invite links. There
  is no public handle behind them, so nothing can be queued.
- **`web_preview(/s/)`**: the web-preview view of a channel. Followable in a
  browser, but it carries no handle the client API can resolve.
- **`deep_link(to_message)`**: a link to one specific message. The handle *is*
  queueable, but the message pointer is lost: the crawler pulls the channel's
  history, not that message.
- **`public_handle(queued)`**: a plain channel reference, the case the snowball
  is built for.
- The remainder are query links (bot start parameters, share links), reserved
  paths, and malformed strings.

The first three categories are the interesting number: they are references the
crawler can see and provably cannot act on.

In [ ]:
SQL_TEXTURL = """
    SELECT m.channel_id, m.message_id, lower(e->>'url') AS url
    FROM messages m, LATERAL jsonb_array_elements(m.entities) e
    WHERE m.date >= :since
      AND m.entities::text LIKE '%t.me/%'
      AND e->>'url' IS NOT NULL
      AND lower(e->>'url') LIKE '%t.me/%'
"""

# Classification applied to each URL above. `FALSE_POSITIVES` holds the reserved
# first-path segments Telegram uses for its own pages (share, addstickers, and so
# on), which look like handles but are not.
#
#     HANDLE_FULL_RE = re.compile(r"[a-z]\w{3,}")   # whole first segment is queueable
#
#     def classify_tme(url):
#         path = url[url.find("t.me/") + 5:].strip().rstrip(").,;:!]")
#         if path == "":                       return "bare_root"
#         if path.startswith("+"):             return "invite(+hash)"
#         parts, first = path.split("/"), first_segment(path)
#         if first == "joinchat":              return "invite(joinchat)"
#         if first in FALSE_POSITIVES:         return "reserved_path"
#         if first == "s" and len(parts) >= 2: return "web_preview(/s/)"
#         if len(parts) >= 2 and is_numeric(parts[1]):
#             return "deep_link(to_message)" if HANDLE_FULL_RE.fullmatch(first) else "deep_link(other)"
#         if "?" in path or "=" in path:
#             return "query_link(bot/start/share)" if HANDLE_FULL_RE.fullmatch(first) else "query_link(other)"
#         if HANDLE_FULL_RE.fullmatch(first):  return "public_handle(queued)"
#         if re.fullmatch(r"[a-z0-9_]+", first): return "handle_rejected(short/digit-lead)"
#         return "other/malformed"

buckets = csv("frontier_link_types.csv")
n = int(buckets["count"].sum())
never = int(buckets.loc[buckets["link_type"].str.startswith(("invite", "web_preview")), "count"].sum())
deep = int(buckets.loc[buckets["link_type"] == "deep_link(to_message)", "count"].sum())
queued = int(buckets.loc[buckets["link_type"] == "public_handle(queued)", "count"].sum())
print(f"t.me references in window: {n:,}")
print(f"  structurally never queueable (invite + web preview): {never:,}  ({never / n:.1%})")
print(f"  deep links to a message (handle queued, message lost): {deep:,}  ({deep / n:.1%})")
print(f"  plain public-handle references (queueable)          : {queued:,}  ({queued / n:.1%})")

fig, ax = plt.subplots(figsize=(5.48, 2.74))
order = buckets.sort_values("count")
colors = ["#b03030" if b.startswith(("invite", "web_preview")) else
          "#c98a00" if b.startswith(("deep_link", "query_link")) else
          "#1f3b6a" for b in order["link_type"]]
ax.barh(order["link_type"], order["count"], color=colors)
ax.set_xscale("log")
ax.set_xlabel("references (log)")
ax.set_title("t.me reference types in collected content (red = never queueable)")
fig.tight_layout()
fig.savefig(FIGURES / "frontier_link_types.png", bbox_inches="tight")
fig.savefig(FIGURES / "frontier_link_types.pdf", bbox_inches="tight")
plt.show()

## Reading

The corpus trades breadth for depth. It holds fewer channels than the predecessor
and roughly a quarter of its channel messages, but about an order of magnitude
more messages per channel, plus layers the predecessor has none of: comments,
archived edits, detected deletions, longitudinal metadata snapshots.

The volume decomposition is what the depth claim rests on. Raw daily volume
rises partly because the snowball keeps adding channels; messages per active
channel is the margin the crawl does not manufacture, and it is the one to quote.

The frontier audit bounds the breadth from the other side. A meaningful share of
the `t.me` references the crawler collected point at private invites or web
previews, which no amount of further crawling can resolve into a channel. That is
a property of Telegram's link surface, not of this crawl.